# 타이타닉 생존 예측 - 2단계: 데이터 전처리

## 프로젝트 진행 상황
- ✅ **1단계 완료**: EDA와 베이스라인 모델 (`01_eda_and_baseline.ipynb`)
- 🔄 **2단계 진행중**: 데이터 전처리 (`02_data_preprocessing.ipynb`) ← **지금 여기**
- ⏳ **3단계 예정**: ML 모델링 및 평가
- ⏳ **4단계 예정**: 최종 제출

## 🎯 이번 노트북의 목표
1. **누락값 처리** - Age, Cabin, Embarked 해결
2. **특성 엔지니어링** - 새로운 의미있는 특성 생성
3. **범주형 변수 인코딩** - 머신러닝 모델이 사용할 수 있도록 변환
4. **데이터 분할** - 훈련/검증 세트 준비
5. **전처리된 데이터 저장** - 다음 단계에서 바로 사용할 수 있도록

> 💡 **실무 팁**: EDA에서 발견한 인사이트를 바탕으로 전처리 전략을 수립합니다!

## 1단계: 환경 설정 및 데이터 로딩

EDA에서 사용한 데이터와 발견한 인사이트를 바탕으로 전처리를 시작합니다.

In [ ]:
# 필수 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 머신러닝 관련 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ 라이브러리 로딩 완료!")
print("📂 데이터 전처리 준비 완료!")

In [ ]:
# 데이터 경로 설정
DATA_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

# 데이터 로딩
print("📊 데이터 로딩 중...")
train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')

print(f"✅ 훈련 데이터: {train_df.shape}")
print(f"✅ 테스트 데이터: {test_df.shape}")

# EDA에서 발견한 주요 인사이트 요약
print("\n🔍 EDA에서 발견한 주요 패턴:")
print("- 여성 생존율 >> 남성 생존율")
print("- 1등실 > 2등실 > 3등실 순으로 생존율 높음")
print("- 어린이의 생존율이 상대적으로 높음")
print("- Age(19.9%), Cabin(77.1%), Embarked(0.2%) 누락값 존재")
print("\n💡 이러한 패턴을 활용해서 전처리 전략을 수립합니다!")

## 2단계: 누락값 처리 전략

EDA에서 확인한 누락값들을 체계적으로 처리합니다. 
각 컬럼의 특성에 맞는 최적의 전략을 사용합니다.

In [ ]:
# 훈련 데이터와 테스트 데이터를 합쳐서 일관된 전처리
# (실무에서 중요: 두 데이터셋이 같은 방식으로 처리되어야 함)

print("🔄 훈련 + 테스트 데이터 결합 (일관된 전처리를 위해)")
train_len = len(train_df)

# 훈련 데이터에 'is_train' 컬럼 추가
train_df['is_train'] = 1
test_df['is_train'] = 0

# 결합 (test에는 Survived 컬럼이 없으므로 추가)
test_df['Survived'] = -1  # 임시값

# 전체 데이터 결합
all_data = pd.concat([train_df, test_df], ignore_index=True)
print(f"✅ 결합된 데이터: {all_data.shape}")

# 누락값 현황 재확인
print(f"\n❗ 누락값 현황:")
missing_cols = all_data.isnull().sum()
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
for col, missing_count in missing_cols.items():
    missing_pct = (missing_count / len(all_data)) * 100
    print(f"- {col}: {missing_count}개 ({missing_pct:.1f}%)")

### 🧑‍⚕️ Age (나이) 누락값 처리
나이는 생존에 중요한 요인이므로 단순히 평균값으로 채우지 않고, 다른 특성들을 고려해서 추정합니다.

In [ ]:
# Age 누락값 처리: 성별 + 객실등급별 중앙값으로 대체
print("🧑‍⚕️ Age 누락값 처리 - 성별 + 객실등급별 중앙값 사용")

# 성별 + 객실등급별 나이 중앙값 계산
age_medians = all_data.groupby(['Sex', 'Pclass'])['Age'].median()
print("📊 성별 + 객실등급별 나이 중앙값:")
print(age_medians)

# 누락값 채우기
def fill_age(row):
    if pd.isna(row['Age']):
        return age_medians[(row['Sex'], row['Pclass'])]
    return row['Age']

all_data['Age'] = all_data.apply(fill_age, axis=1) # all_data.apply(fill_age, axis=1)는 "데이터프레임의 모든 행을 하나씩 
                        #fill_age 함수에 넣어서 실행하고, 그 결과를 모아서 새로운 Age 컬럼을 만들어줘" 라는 강력하고 효율적인 명령입니다.

print(f"\n✅ Age 누락값 처리 완료!")
print(f"처리 후 Age 누락값: {all_data['Age'].isnull().sum()}개")

# 처리 결과 확인
print(f"\n📈 처리된 나이 분포:")
print(f"- 평균: {all_data['Age'].mean():.1f}세")
print(f"- 중앙값: {all_data['Age'].median():.1f}세")
print(f"- 범위: {all_data['Age'].min():.0f}세 ~ {all_data['Age'].max():.0f}세")

### 🚢 Embarked (탑승지) 누락값 처리
누락값이 매우 적으므로(0.2%) 최빈값으로 간단히 처리합니다.

In [ ]:
# Embarked 누락값 처리: 최빈값으로 대체
print("🚢 Embarked 누락값 처리")

# 현재 분포 확인
embarked_counts = all_data['Embarked'].value_counts()
print("📊 탑승지별 분포:")
print(embarked_counts)

# 최빈값으로 누락값 채우기
most_common_embarked = all_data['Embarked'].mode()[0]
all_data['Embarked'].fillna(most_common_embarked, inplace=True)

print(f"\n✅ Embarked 누락값 처리 완료!")
print(f"누락값을 '{most_common_embarked}'로 대체")
print(f"처리 후 Embarked 누락값: {all_data['Embarked'].isnull().sum()}개")

### 🏠 Cabin (객실) 누락값 처리
Cabin은 77% 누락값이므로, 누락 여부 자체를 새로운 특성으로 활용합니다.

In [ ]:
# Cabin 처리: 누락값 여부를 새로운 특성으로 활용
print("🏠 Cabin 처리 - 누락값 정보를 특성으로 활용")

# Cabin이 있는지 없는지를 새로운 특성으로 생성
all_data['Has_Cabin'] = all_data['Cabin'].notna().astype(int)

# Cabin이 있는 경우와 없는 경우의 생존율 비교 (훈련 데이터만)
train_data_temp = all_data[all_data['is_train'] == 1].copy()
cabin_survival = train_data_temp.groupby('Has_Cabin')['Survived'].agg(['count', 'mean']).round(3)
cabin_survival.columns = ['승객 수', '생존율']

print("📊 객실 정보 유무별 생존율:")
print("0 = 객실 정보 없음, 1 = 객실 정보 있음")
print(cabin_survival)

# Cabin에서 첫 글자(데크) 정보 추출
all_data['Cabin_Deck'] = all_data['Cabin'].str[0]
all_data['Cabin_Deck'].fillna('Unknown', inplace=True)

print(f"\n✅ Cabin 처리 완료!")
print(f"- Has_Cabin: 객실 정보 유무 (0 또는 1)")
print(f"- Cabin_Deck: 객실 데크 정보 ({all_data['Cabin_Deck'].nunique()}개 카테고리)")

# 원본 Cabin 컬럼 제거 (너무 많은 누락값으로 직접 사용하기 어려움)
all_data.drop('Cabin', axis=1, inplace=True)
print(f"- 원본 Cabin 컬럼 제거")

## 3단계: 특성 엔지니어링 (Feature Engineering)

EDA에서 발견한 패턴을 바탕으로 새로운 의미있는 특성들을 생성합니다.

In [ ]:
# 1. 가족 규모 특성 생성
print("👨‍👩‍👧‍👦 가족 규모 특성 생성")
all_data['Family_Size'] = all_data['SibSp'] + all_data['Parch'] + 1  # 본인 포함

# 가족 규모를 카테고리로 분류
def categorize_family_size(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'

all_data['Family_Size_Group'] = all_data['Family_Size'].apply(categorize_family_size)

# 2. 나이대 그룹 생성 (EDA에서 사용한 것과 동일)
print("👶👴 나이대 그룹 생성")
def categorize_age(age):
    if age < 18:
        return 'Child'
    elif age < 30:
        return 'Young_Adult'
    elif age < 50:
        return 'Middle_Age'
    else:
        return 'Senior'

all_data['Age_Group'] = all_data['Age'].apply(categorize_age)

# 3. 요금 범위 생성
print("💰 요금 범위 특성 생성")
# 요금을 4분위수로 나누기
all_data['Fare_Range'] = pd.qcut(all_data['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very_High'])

# 4. 제목(Title) 추출 - 이름에서 사회적 지위 파악
print("🎩 이름에서 제목(Title) 추출")
all_data['Title'] = all_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# 제목 그룹화 (빈도가 낮은 것들은 합치기)
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss', 
    'Mrs': 'Mrs',
    'Master': 'Master'
}

# 나머지는 모두 'Other'로 분류
all_data['Title'] = all_data['Title'].map(title_mapping).fillna('Other')

print(f"\n✅ 새로운 특성 생성 완료!")
print(f"- Family_Size: 가족 규모 ({all_data['Family_Size'].min()}-{all_data['Family_Size'].max()})")
print(f"- Family_Size_Group: 가족 그룹 ({all_data['Family_Size_Group'].unique()})")
print(f"- Age_Group: 나이대 그룹 ({all_data['Age_Group'].unique()})")
print(f"- Fare_Range: 요금 범위 ({all_data['Fare_Range'].unique()})")
print(f"- Title: 제목 ({all_data['Title'].unique()})")

## 4단계: 데이터 인코딩 및 최종 준비

머신러닝 모델이 사용할 수 있도록 범주형 변수를 숫자로 변환하고, 최종 데이터셋을 준비합니다.

**이론적으로는** Embarked , Sex처럼 순서가 없는 특성에는 LabelEncoder가 부적합해 보입니다. 하지만 여기에는 몇가지 실용적인 이유가 있습니다.

1. 트리 기반 모델은 순서에 덜 민감하다. **(가장 중요한 이유)**

2. 코드의 간결성과 통일성
    - 이 프로젝트에서는 여러 범주형 변수를 한 번에 처리해야함. LabelEncoder는 for루프를 통해 모든 범주형 특성을 일관된 방식으로 간단하게 숫자로 
    바꿀 수 있음.

만약 우리가 선형 회귀(Linear Regression)나 로지스틱 회귀(Logistic Regression)처럼 숫자의 크기 자체가 모델의 성능에 직접적인 영향을 주는 모델을 주력으로 사용한다면, LabelEncoder 대신 원-핫 인코딩을 사용하는 것이 훨씬 더 좋은 선택이 됩니다.

In [ ]:
# 범주형 변수들을 숫자로 인코딩
print("🔢 범주형 변수 인코딩")

# Label Encoding할 컬럼들
categorical_columns = ['Sex', 'Embarked', 'Family_Size_Group', 'Age_Group', 
                      'Fare_Range', 'Title', 'Cabin_Deck']

# 각 범주형 변수에 대해 Label Encoding 적용
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    all_data[col + '_encoded'] = le.fit_transform(all_data[col])
    #  나중에 모델 예측 후 결과를 다시 원래의 문자로 되돌리거나 새로운 데이터에 동일한 규칙을 적용할 때 이 저장된 인코더를 꺼내 쓸 수 있기 때문에 사용.
    label_encoders[col] = le
    
    # 인코딩 결과 확인
    print(f"- {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\n✅ 인코딩 완료! {len(categorical_columns)}개 변수 처리")

In [ ]:
# 최종 특성 선택 및 데이터셋 분할
print("🎯 최종 특성 선택 및 데이터 분할")

# ML 모델에 사용할 특성들 선택
feature_columns = [
    # 기본 특성
    'Pclass', 'Age', 'Fare', 'Family_Size', 'Has_Cabin',
    
    # 인코딩된 범주형 특성
    'Sex_encoded', 'Embarked_encoded', 'Family_Size_Group_encoded',
    'Age_Group_encoded', 'Fare_Range_encoded', 'Title_encoded', 'Cabin_Deck_encoded'
]

# 훈련 데이터와 테스트 데이터 분할
train_processed = all_data[all_data['is_train'] == 1].copy()
test_processed = all_data[all_data['is_train'] == 0].copy()

# 특성과 타겟 분리
X_train_full = train_processed[feature_columns]
y_train_full = train_processed['Survived']
X_test_final = test_processed[feature_columns]

# 검증용 데이터 분할 (80% 훈련, 20% 검증)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

print(f"✅ 데이터 분할 완료!")
print(f"- 훈련 데이터: {X_train.shape}")
print(f"- 검증 데이터: {X_val.shape}")
print(f"- 테스트 데이터: {X_test_final.shape}")
print(f"- 사용 특성: {len(feature_columns)}개")

# 특성 목록 출력
print(f"\n📊 사용할 특성들:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# 전처리된 데이터 저장
print("💾 전처리된 데이터 저장")

import os
os.makedirs(PROCESSED_PATH, exist_ok=True)

# 훈련/검증/테스트 데이터 저장
X_train.to_csv(PROCESSED_PATH + 'X_train.csv', index=False)
X_val.to_csv(PROCESSED_PATH + 'X_val.csv', index=False)
y_train.to_csv(PROCESSED_PATH + 'y_train.csv', index=False)
y_val.to_csv(PROCESSED_PATH + 'y_val.csv', index=False)
X_test_final.to_csv(PROCESSED_PATH + 'X_test.csv', index=False)

# 테스트 데이터의 PassengerId도 저장 (제출시 필요)
test_ids = test_processed['PassengerId']
test_ids.to_csv(PROCESSED_PATH + 'test_ids.csv', index=False)

# 특성 이름 저장
pd.Series(feature_columns).to_csv(PROCESSED_PATH + 'feature_names.csv', index=False)

print(f"✅ 저장 완료! 저장 위치: {PROCESSED_PATH}")
print(f"📁 저장된 파일들:")
saved_files = [
    'X_train.csv', 'X_val.csv', 'y_train.csv', 'y_val.csv', 
    'X_test.csv', 'test_ids.csv', 'feature_names.csv'
]
for file in saved_files:
    print(f"  - {file}")

print(f"\n🚀 다음 단계: 03_modeling_and_evaluation.ipynb에서 실제 ML 모델 훈련!")

---

## 🎉 데이터 전처리 완료! 

### ✅ 완료된 작업들:

#### 1. **누락값 처리** ✓
- **Age**: 성별+객실등급별 중앙값으로 대체
- **Embarked**: 최빈값('S')으로 대체
- **Cabin**: 누락값 정보를 새로운 특성으로 활용

#### 2. **특성 엔지니어링** ✓
- **Family_Size**: SibSp + Parch + 1 (가족 규모)
- **Family_Size_Group**: Alone, Small, Large (가족 그룹)
- **Age_Group**: Child, Young_Adult, Middle_Age, Senior
- **Fare_Range**: Low, Medium, High, Very_High (요금 범위)
- **Title**: Mr, Mrs, Miss, Master, Other (이름에서 추출)
- **Has_Cabin**: 객실 정보 유무 (0/1)
- **Cabin_Deck**: 객실 데크 정보

#### 3. **데이터 인코딩** ✓
- 모든 범주형 변수를 숫자로 변환
- Label Encoding 사용

#### 4. **데이터 분할** ✓
- 훈련 데이터: 80% (712명)
- 검증 데이터: 20% (179명)  
- 테스트 데이터: 418명

#### 5. **데이터 저장** ✓
- 전처리된 모든 데이터를 `data/processed/` 폴더에 저장
- 다음 노트북에서 바로 사용 가능

### 🚀 다음 단계 예고: `03_modeling_and_evaluation.ipynb`
- 다양한 ML 알고리즘 비교 (로지스틱 회귀, 랜덤 포레스트, SVM 등)
- 하이퍼파라미터 튜닝
- 교차검증을 통한 모델 평가
- 특성 중요도 분석

**실무 팁**: 이렇게 체계적으로 전처리를 분리해놓으면, 나중에 다른 모델을 시도할 때도 일관된 데이터를 사용할 수 있어 매우 효율적입니다!